In [ ]:
# Copyright 2023 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

[**This is an Experimental release. Experiments are focused on validating a prototype and are not guaranteed to be released. Experiments are covered by the [Pre-GA Offerings Terms](https://cloud.google.com/terms/service-terms) of the Google Cloud Platform Terms of Service. They are not intended for production use or covered by any SLA,  support obligation, or deprecation policy and may be subject to backward-incompatible changes.**]

# AutoSxS Experimental Release Guide

AutoSxS is an LLM evaluation tool, which enables users to compare the performance of Google-first-party LLMs. As part of AutoSxS's experimental release, we only support comparing models against a summarization task. We will support more tasks in the future.

An example use case of AutoSxS may be that you are a data scientist developing a summarization tool specific to your company's data. You currently have a working solution, using Generative AI Studio's text-bison, but you would like to know if smaller models or fine-tuned models would perform better than Bison (or about as well, for cheaper).

In this guide, we will walk you through the full process of using AutoSxS, including guidance on setting up your notebook environment, preparing your dataset, and finally running the AutoSxS pipeline.


## Setup

In this section we will configure the notebook for GCP usage and install/import dependencies.

### Install Dependencies

First, we install the latest version of the Google Cloud Pipeline Components (GCPC) SDK and restart the notebook kernel to find the packages.

Note: You may get a message saying "Your session crashed for an unknown reason." or messages about "pip's dependency resolver", this is expected. Once this cell has finished running, continue on.

In [ ]:
import os

# Google Cloud Notebook
if os.path.exists("/opt/deeplearning/metadata/env_version"):
    USER_FLAG = "--user"
else:
    USER_FLAG = "";

! pip3 install --upgrade --force-reinstall $USER_FLAG google-cloud-aiplatform==1.29.0

if not os.getenv("IS_TESTING"):
    # Automatically restart kernel after installs
    import IPython

    app = IPython.Application.instance()
    app.kernel.do_shutdown(True)


### Setting the Project ID
If you don't know your project ID, leave the field blank, and the following cells may be able to find it. Optionally, you may set a service account in the cell below.



In [ ]:
PROJECT_ID = "[your-project-id]"  # @param {type:"string"}
PROJECT_ID = "YOUR_PROJECT_ID"  # @param {type:"string"}


### Setting the Region
You may change the `REGION` variable, which is used for operations
throughout the rest of this notebook.  Below are regions supported for AutoSxS. We recommend that you choose the region closest to you where you have sufficient GPU/TPU quota. If you do not set a region, we will default to using `europe-west4`.

- Americas: `us-central1`
- Europe: `europe-west4`
- Asia Pacific: `asia-southeast1`

Learn more about [Vertex AI regions](https://cloud.google.com/vertex-ai/docs/general/locations)

In [ ]:
REGION = "[your-region]"  # @param {type: "string"}
REGION = "europe-west4"

In [ ]:
if PROJECT_ID == "" or PROJECT_ID is None or PROJECT_ID == "[your-project-id]":
    # Get your GCP project id from gcloud
    shell_output = ! gcloud config list --format 'value(core.project)' 2>/dev/null
    PROJECT_ID = shell_output[0]
    print("Project ID:", PROJECT_ID)

if REGION == "[your-region]":
    REGION = "europe-west4"

! gcloud config set project $PROJECT_ID

### UUID

We define a UUID generation function to avoid resource name collisions on resources created within the notebook.

In [ ]:
import random
import string


def generate_uuid(length: int = 8) -> str:
    """Generate a uuid of a specifed length (default=8)."""
    return "".join(random.choices(string.ascii_lowercase + string.digits, k=length))


UUID = generate_uuid()

### Authenticate your Google Cloud account

In [ ]:
# If you are running this notebook in Colab, run this cell and follow the
# instructions to authenticate your GCP account. This provides access to your
# Cloud Storage bucket and lets you submit training jobs and prediction
# requests.

import os
import sys

# If on Vertex AI Workbench, then don't execute this code
IS_COLAB = "google.colab" in sys.modules
if not os.path.exists("/opt/deeplearning/metadata/env_version") and not os.getenv(
    "DL_ANACONDA_HOME"
):
    if "google.colab" in sys.modules:
        from google.colab import auth as google_auth

        google_auth.authenticate_user()

    # If you are running this notebook locally, replace the string below with the
    # path to your service account key and run this cell to authenticate your GCP
    # account.
    elif not os.getenv("IS_TESTING"):
        %env GOOGLE_APPLICATION_CREDENTIALS '[your-service-account-key-path]'

### Create a Cloud Storage bucket

We use the bucket to store intermediate data and inputs to the AutoSxS pipeline.


In [ ]:
BUCKET_URI = "gs://[your-bucket-name]"  # @param {type:"string"}
BUCKET_URI = "gs://YOUR_GCS_BUCKET_NAME"  # @param {type:"string"}


Create the bucket if it doesn't already exist.

In [ ]:
if BUCKET_URI == "" or BUCKET_URI is None or BUCKET_URI == "gs://[your-bucket-name]":
    BUCKET_URI = "gs://" + PROJECT_ID + "aip-" + UUID

! gsutil ls -b $BUCKET_URI || gsutil mb -l $REGION $BUCKET_URI

### Import remaining libraries

In [ ]:
import json
import os
import urllib
import uuid

from google.cloud import aiplatform
import pandas as pd

## Generate AutoSxS Input Dataset

Below we create our dataset, specifying the set of prompts to evaluate on.

####**Note: For experimentation, we request that users provide only a few prompts. For best results, we recommend users input 100-500 examples. There are diminishing returns past 400 examples, and if more than 500 examples are provided, AutoSxS will raise a validation error.**

> Indented block



In [ ]:
prompts = [
    "The earliest large language models were built with recurrent architectures such as the long short-term memory (LSTM) (1997). After AlexNet (2012) demonstrated the effectiveness of large neural networks for image recognition, researchers applied large neural networks to other tasks. In 2014, two main techniques were proposed.",
    "In 2016, Google Translate changed its technology from statistical machine translation to neural machine translation. It was a seq2seq with LSTM and attention. It took them 9 months to reach a higher level of performance than the previous system built over 10 years.",
    "The 2017 paper 'Attention is all you need' abstracted out the attention mechanism from 2014 paper by Bahdanau et al., and constructed the Transformer architecture around the attention mechanism.",
]

examples = pd.DataFrame({
    'content': prompts
})

Alternatively, you can load your own JSONL dataset from GCS. Just note that you will need to change the `prompt_column` field later in the notebook to match the column name of your GCS dataset.

Input data is expected to be in JSONL format, where each line is a row in the dataset. The dataset can consist of multiple columns, but only the `prompt_column` will be used for AutoSxS.

**Example:**
```
{"content": "The earliest large language models were built with recurrent architectures such as the long short-term memory (LSTM) (1997). After AlexNet (2012) demonstrated the effectiveness of large neural networks for image recognition, researchers applied large neural networks to other tasks. In 2014, two main techniques were proposed."}
{"content": "In 2016, Google Translate changed its technology from statistical machine translation to neural machine translation. It was a seq2seq with LSTM and attention. It took them 9 months to reach a higher level of performance than the previous system built over 10 years."}
{"content": "The 2017 paper 'Attention is all you need' abstracted out the attention mechanism from 2014 paper by Bahdanau et al., and constructed the Transformer architecture around the attention mechanism."}
```

In [ ]:
# # Uncomment to read from GCS.
# GCS_PATH = 'gs://[path-to-your-dataset]'
# examples = pd.read_json(GCS_PATH, lines=True)

We next upload our final dataset to GCS to be used as input for AutoSxS.

In [ ]:
# Truncate examples to fit within the 500 examples limit.
examples = examples.head(500)

assert len(examples) <= 500, 'Number of examples must be less than 500.'

examples.to_json('data.json', orient='records', lines=True)
! gsutil cp data.json $BUCKET_URI/temp/data.json
DATASET = f'{BUCKET_URI}/temp/data.json'

## AutoSxS Job

In order to run AutoSxS, we need to define a pipeline job with the following parameters. Note that many are optional.

Required parameters:
```
    project: (str) The GCP project that runs the pipeline components.

    location: (str) The GCP region to run pipeline components in.
    
    base_model_a_name: (str) The base instruction-tuned model used by Model A.
      Valid options are 'PALM_TINY', 'GECKO', 'OTTER', 'BISON', 'T5_SMALL',
      'T5_LARGE', 'T5_XL', and 'T5_XXL'.

    base_model_b_name: (str) The base instruction-tuned model used by Model B.
      Valid options are 'PALM_TINY', 'GECKO', 'OTTER', 'BISON', 'T5_SMALL',
      'T5_LARGE', 'T5_XL', and 'T5_XXL'.

    prompt_dataset: (str) A GCS path to a JSONL dataset, containing prompts to
      use for evaluation.

    prompt_column: (str) The column name, containing prompts.
```

Optional parameters:
```
    instruction: (Optional[str]) An instruction to prepend to all prompts. If no
      instruction is given, one will be generated based on the task.

    prompt_sequence_length: (Optional[int]) The maximum number of tokens allowed
      in a prompt.

    target_sequence_length: (Optional[int]) The maximum number of tokens allowed
      in a response.

    importer_machine_type: (Optional[str]) The machine type to use for
      importing/preprocessing the data.

    use_gpu_defaults: (Optional[bool]) If True, any components that use
      accelerators will default to using GPUs instead of TPUs.

    model_a_checkpoint_override: (Optional[str]) The checkpoint to use for
      Model A if using a tuned model. If no checkpoint is specified, the base
      image will be used.

    model_b_checkpoint_override: (Optional[str]) The checkpoint to use for
      Model B if using a tuned model. If no checkpoint is specified, the base
      image will be used.

    accelerator_type_model_a_override: (Optional[str]) The accelerator type to
      use for Model A, when accelerators are required. Valid options are:
      'TPU_V2', 'TPU_V3', 'NVIDIA_TESLA_A100', and 'NVIDIA_A100_80GB'.

    accelerator_type_model_b_override: (Optional[str]) The accelerator type to
      use for Model B, when accelerators are required. Valid options are:
      'TPU_V2', 'TPU_V3', 'NVIDIA_TESLA_A100', and 'NVIDIA_A100_80GB'.
      
    accelerator_count_model_a_override: (Optional[int]) The number of
      accelerators to use for Model A, when accelerators are required. GPUs are
      counted by chip, TPUs are counted by core.

    accelerator_count_model_b_override: (Optional[int]) The number of
      accelerators to use for Model A, when accelerators are required. GPUs are
      counted by chip, TPUs are counted by core.
```

The following code starts a Vertex Pipeline job, viewable from the Vertex UI. The logs here will include to the URL to the current pipeline, so you can follow the pipline progress and access/view pipeline outputs.

In [ ]:
template_uri = 'https://us-kfp.pkg.dev/ml-pipeline/optimization/autosxs/sha256:57ba2393ff66253eabaf37f4abc139288bb23e54a75c4bb1d421c24ec6d4a03c'
display_name = f'autosxs-experimental-{generate_uuid()}'
parameters = {
    'base_model_a_name': 'GECKO',
    'base_model_b_name': 'OTTER',
    'location': REGION,
    'project': PROJECT_ID,
    'prompt_dataset': DATASET,
    'prompt_column': 'content',
}

# Advanced config.
parameters.update({
    'instruction': 'Summarize in under 64 words.',
    'prompt_sequence_length': 512,
    'target_sequence_length': 64,
    'importer_machine_type': 'e2-highmem-8',
    'use_gpu_defaults': False,
    'model_a_checkpoint_override': '',
    'model_b_checkpoint_override': '',
    'accelerator_type_model_a_override': None,
    'accelerator_count_model_a_override': None,
    'accelerator_type_model_b_override': None,
    'accelerator_count_model_b_override': None,
})

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)
job = aiplatform.PipelineJob(
    job_id=display_name,
    display_name=display_name,
    pipeline_root=os.path.join(BUCKET_URI, display_name),
    template_path=template_uri,
    parameter_values=parameters,
)
job.run()

After the job completes, we can load AutoSxS's judgements into a DataFrame.

In [ ]:
# To use an existing pipeline, override job using the line below.
# job = aiplatform.PipelineJob.get('projects/[PROJECT_NUMBER]/locations/[REGION]/pipelineJobs/[PIPELINE_RUN_NAME]')

for details in job.task_details:
  if details.task_name == 'autosxs-arbiter':
    break

judgements_uri = details.outputs['judgements'].artifacts[0].uri
judgements_df = pd.read_json(judgements_uri, lines=True)
judgements_df.head()

## Run the Arbiter Directly

If you wish to bypass the data inference step of AutoSxS and only compare the the predictions of two arbitrary models (run possibly outside of AutoSxS and not necessarily Google-first-party models), you may do so by calling our Arbiter component directly.

This can be accomplished either by using the Arbiter component or by using a CustomJob like in the cells below.

In [ ]:
# Define prompts and predictions.
prompts = [
    "The earliest large language models were built with recurrent architectures such as the long short-term memory (LSTM) (1997). After AlexNet (2012) demonstrated the effectiveness of large neural networks for image recognition, researchers applied large neural networks to other tasks. In 2014, two main techniques were proposed.",
    "In 2016, Google Translate changed its technology from statistical machine translation to neural machine translation. It was a seq2seq with LSTM and attention. It took them 9 months to reach a higher level of performance than the previous system built over 10 years.",
    "The 2017 paper 'Attention is all you need' abstracted out the attention mechanism from 2014 paper by Bahdanau et al., and constructed the Transformer architecture around the attention mechanism.",
]
predictions_a = [
    "RNNs are old news, they can't remember what they had for breakfast this morning.",
    "I can't believe I spent all those years studying statistical methods.",
    "The paper 'Attention is all you need' discussed the Transformer architecture.",
]
predictions_b = [
    "Recurrent architectures were used initially, but researches found large neural architectures to be better.",
    "Google translate achieved better results with neural machine translation very quickly.",
    "Optimus Prime would lose in a one-on-one fight against Megatron.",
]

# Create the prediction files.
with open('preds_a.jsonl', 'w') as f:
  for prompt, pred in zip(prompts, predictions_a):
    json.dump({
        "inputs": {"inputs_pretokenized": prompt},
        "prediction": pred,
    }, f)
    f.write('\n')

with open('preds_b.jsonl', 'w') as f:
  for prompt, pred in zip(prompts, predictions_b):
    json.dump({
        "inputs": {"inputs_pretokenized": prompt},
        "prediction": pred,
    }, f)
    f.write('\n')

In [ ]:
# Upload predictions to GCS.
PREDS_A = f'{BUCKET_URI}/temp/preds_a/preds_a.jsonl-00000-of-00001'
PREDS_B = f'{BUCKET_URI}/temp/preds_b/preds_b.jsonl-00000-of-00001'
! gsutil cp preds_a.jsonl $PREDS_A
! gsutil cp preds_b.jsonl $PREDS_B

Use a CustomJob to run the Arbiter component directly. The outputs will be  stored in `output_dir`.

In [ ]:
display_name = f'arbiter-experimental-{generate_uuid()}'
output_dir = f'{BUCKET_URI}/{display_name}_output/'
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)
aiplatform.CustomJob(
    project=PROJECT_ID,
    location=REGION,
    display_name=display_name,
    worker_pool_specs=[
        {
            'machine_spec': {
                'machine_type': 'n1-standard-4',
            },
            'replica_count': 1,
            'container_spec': {
                'image_uri': 'us-docker.pkg.dev/vertex-ai-restricted/rlhf/rlhf_autosxs@sha256:803f292baf1e723e8dfce8ef60670af1238b430170db7f65ba2811711005ddd2',
                'args': [
                    '--remote_coredump_enabled',
                    '--', 'arbiter',
                    '--model_a_prediction_dir', f'{BUCKET_URI}/temp/preds_a/',
                    '--model_b_prediction_dir', f'{BUCKET_URI}/temp/preds_b/',
                    '--instruction', 'Summarize in less than 64 words.',
                    '--output_dir', output_dir,
                ],
            },
        }
    ],
).run()